# Neural Network Regularization: Dropout vs L2 (Weight Decay)

## Overview
This notebook trains a simple Feedforward Neural Network (FFNN) on the MNIST dataset in **three configurations**:

| Variant | Dropout | L2 (Weight Decay) |
|---------|---------|-------------------|
| **Baseline** (no regularization) | ✗ | ✗ |
| **Dropout** | ✓ (p = 0.5) | ✗ |
| **L2 Regularization** | ✗ | ✓ (λ = 1e-3) |

**Goal:** Compare training and testing accuracies across the three variants and discuss how regularization techniques help improve model generalization.

**Dataset:** MNIST, consisting of 70,000 grayscale images (28×28) of handwritten digits (0-9).
- Training: 48,000 samples
- Validation: 12,000 samples
- Test: 10,000 samples

In [ ]:
# Imports & Setup
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import warnings
import os
import copy
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Data Loading & Exploration

We load the MNIST dataset, apply a train/validation split (80/20), and inspect the data.

In [ ]:
# Data Loading
transform = transforms.Compose([transforms.ToTensor()])

data_root = './data'
train_dataset = datasets.MNIST(root=data_root, train=True,  transform=transform, download=True)
test_dataset  = datasets.MNIST(root=data_root, train=False, transform=transform, download=True)

# 80/20 train-validation split
train_size = int(0.8 * len(train_dataset))
val_size   = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(
    train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

batch_size   = 64
pin_memory   = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  pin_memory=pin_memory)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, pin_memory=pin_memory)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, pin_memory=pin_memory)

print(f'Training   samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Test       samples: {len(test_dataset)}')
print(f'Batch size: {batch_size}')

In [ ]:
# Sample Images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample MNIST Images', fontsize=14, fontweight='bold')

sample_batch, sample_labels = next(iter(train_loader))
for idx in range(10):
    ax = axes[idx // 5, idx % 5]
    ax.imshow(sample_batch[idx].squeeze().numpy(), cmap='gray')
    ax.set_title(f'Label: {sample_labels[idx].item()}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 2. Model Architecture

We define **three model variants** sharing the same base architecture:

```
Input (784) → Linear(784→256) → ReLU → [Dropout?]
            → Linear(256→128) → ReLU → [Dropout?]
            → Linear(128→64)  → ReLU → [Dropout?]
            → Linear(64→10)   → Output (Logits)
```

| Variant | Dropout (p) | Weight Decay (λ) |
|---------|-------------|------------------|
| Baseline | 0.0 | 0 |
| Dropout | 0.5 | 0 |
| L2 Regularization | 0.0 | 1e-3 |

### Why this architecture?
- **Wider hidden layers (256, 128, 64):** Makes the network intentionally over-parameterized so the effect of regularization is clearly visible.
- **No BatchNorm:** Omitted on purpose so that regularization effects are isolated.
- **High dropout (0.5):** The classic Hinton recommendation; forces the network to learn redundant representations.
- **L2 via weight_decay:** Penalizes large weights, preventing any single neuron from dominating.

In [ ]:
# Model Definitions

class FFNN(nn.Module):
    """Feedforward Neural Network with optional Dropout."""
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)

# Instantiate Three Models
model_configs = {
    'Baseline (No Reg)': {'dropout': 0.0, 'weight_decay': 0.0},
    'Dropout (p=0.5)':   {'dropout': 0.5, 'weight_decay': 0.0},
    'L2 (λ=1e-3)':       {'dropout': 0.0, 'weight_decay': 1e-3},
}

models = {}
for name, cfg in model_configs.items():
    m = FFNN(dropout_rate=cfg['dropout']).to(device)
    models[name] = m
    n_params = sum(p.numel() for p in m.parameters())
    print(f'{name:25s} → {n_params:,} parameters  (dropout={cfg["dropout"]}, wd={cfg["weight_decay"]})')

## 3. Training Infrastructure

We train each model for **20 epochs** using the Adam optimizer and CrossEntropyLoss. Each model is trained independently from scratch.

**Key details:**
- No learning-rate scheduler (keeps comparison fair)
- Early stopping with patience = 5 epochs
- Same data loaders / random seed for all three runs

In [ ]:
# Training & Evaluation Helpers
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = (images - MNIST_MEAN) / MNIST_STD
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = (images - MNIST_MEAN) / MNIST_STD
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

print('Helper functions defined.')

In [ ]:
# Main Training Loop
NUM_EPOCHS = 20
PATIENCE   = 5
LR         = 1e-3
loss_fn    = nn.CrossEntropyLoss()

# Store history for every model
histories = {}

for name, cfg in model_configs.items():
    print(f'\n{"="*70}')
    print(f'Training: {name}')
    print(f'{"="*70}')

    model = models[name]
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=cfg['weight_decay'])

    best_val_loss = float('inf')
    best_state    = None
    no_improve    = 0

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, loss_fn)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        marker = ''
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            marker = ' ✓'
        else:
            no_improve += 1

        print(f'  Epoch {epoch:2d}/{NUM_EPOCHS} │ '
              f'Train Loss {tr_loss:.4f}  Acc {tr_acc:6.2f}% │ '
              f'Val Loss {vl_loss:.4f}  Acc {vl_acc:6.2f}%{marker}')

        if no_improve >= PATIENCE:
            print(f'  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
            break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)
    histories[name] = history

print(f'\n{"="*70}')
print('All models trained.')

## 4. Training Curves: Loss & Accuracy

Side-by-side comparison of training and validation curves for all three models.

In [ ]:
# Plot Style
plt.rcParams.update({
    'figure.facecolor': '#ffffff',
    'axes.facecolor':   '#ffffff',
    'axes.edgecolor':   '#d0d7de',
    'axes.labelcolor':  '#24292f',
    'text.color':       '#24292f',
    'xtick.color':      '#57606a',
    'ytick.color':      '#57606a',
    'grid.color':       '#d0d7de',
    'legend.facecolor': '#f6f8fa',
    'legend.edgecolor': '#d0d7de',
    'font.family':      'sans-serif',
    'font.size':        11,
})

COLORS = {
    'Baseline (No Reg)': ('#d73a49', '#ff8585'),   # Red tones for Light Theme
    'Dropout (p=0.5)':   ('#0366d6', '#8dc1fc'),   # Blue tones for Light Theme
    'L2 (λ=1e-3)':       ('#28a745', '#85ea9c'),   # Green tones for Light Theme
}

print('Plot style configured.')

In [ ]:
# Training vs Validation Loss
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training vs Validation Loss', fontsize=16, fontweight='bold', y=1.02)

for ax, (name, hist) in zip(axes, histories.items()):
    c_train, c_val = COLORS[name]
    epochs = range(1, len(hist['train_loss']) + 1)
    ax.plot(epochs, hist['train_loss'], '-o', color=c_train, markersize=4, label='Train Loss', linewidth=2)
    ax.plot(epochs, hist['val_loss'],   '-s', color=c_val,   markersize=4, label='Val Loss',   linewidth=2)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: loss_curves.png')

In [ ]:
# Training vs Validation Accuracy
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training vs Validation Accuracy', fontsize=16, fontweight='bold', y=1.02)

for ax, (name, hist) in zip(axes, histories.items()):
    c_train, c_val = COLORS[name]
    epochs = range(1, len(hist['train_acc']) + 1)
    ax.plot(epochs, hist['train_acc'], '-o', color=c_train, markersize=4, label='Train Acc', linewidth=2)
    ax.plot(epochs, hist['val_acc'],   '-s', color=c_val,   markersize=4, label='Val Acc',   linewidth=2)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_curves.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: accuracy_curves.png')

## 5. Test Set Evaluation

We evaluate each model on the held-out **10,000 test samples** and produce a comparison table.

In [ ]:
# Test Evaluation
results = []

for name in model_configs:
    model = models[name]
    te_loss, te_acc = evaluate(model, test_loader, loss_fn)

    # Final training accuracy (last epoch)
    tr_acc = histories[name]['train_acc'][-1]
    vl_acc = histories[name]['val_acc'][-1]

    # Best validation accuracy
    best_vl_acc = max(histories[name]['val_acc'])

    # Generalization gap = Train Acc - Test Acc
    gap = tr_acc - te_acc

    results.append({
        'Model': name,
        'Train Acc (%)': round(tr_acc, 2),
        'Val Acc (%)':   round(best_vl_acc, 2),
        'Test Acc (%)':  round(te_acc, 2),
        'Test Loss':     round(te_loss, 4),
        'Gap (Train-Test)': round(gap, 2),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Comparison Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))

model_names = results_df['Model'].tolist()
train_accs  = results_df['Train Acc (%)'].tolist()
test_accs   = results_df['Test Acc (%)'].tolist()

x = np.arange(len(model_names))
width = 0.30

bars1 = ax.bar(x - width/2, train_accs, width, label='Train Accuracy',
               color=[COLORS[n][0] for n in model_names], alpha=0.85, edgecolor='grey', linewidth=0.5)
bars2 = ax.bar(x + width/2, test_accs,  width, label='Test Accuracy',
               color=[COLORS[n][1] for n in model_names], alpha=0.85, edgecolor='grey', linewidth=0.5)

# Annotate bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Training vs Test Accuracy: Regularization Comparison',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(90, 101)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: accuracy_comparison.png')

In [ ]:
# Generalization Gap Bar Chart
fig, ax = plt.subplots(figsize=(8, 5))

gaps = results_df['Gap (Train-Test)'].tolist()
bar_colors = [COLORS[n][0] for n in model_names]

bars = ax.bar(model_names, gaps, color=bar_colors, alpha=0.85, edgecolor='grey', linewidth=0.5, width=0.5)

for bar, gap in zip(bars, gaps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{gap:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Generalization Gap (%)', fontsize=12)
ax.set_title('Generalization Gap (Train Acc - Test Acc)',
             fontsize=14, fontweight='bold')
ax.axhline(y=0, color='#57606a', linestyle='--', linewidth=0.8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('generalization_gap.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: generalization_gap.png')

## 6. Confusion Matrices

Confusion matrices for each model on the test set, side-by-side.

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Confusion Matrices (Test Set)', fontsize=16, fontweight='bold', y=1.02)

for ax, name in zip(axes, model_configs):
    model = models[name]
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = (images - MNIST_MEAN) / MNIST_STD
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    # Normalize
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues', ax=ax,
                xticklabels=range(10), yticklabels=range(10),
                cbar_kws={'label': 'Accuracy (%)'})
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: confusion_matrices.png')

## 7. Per-Class Accuracy Comparison

In [ ]:
# Per-Class Accuracy
fig, ax = plt.subplots(figsize=(14, 6))

for i, name in enumerate(model_configs):
    model = models[name]
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = (images - MNIST_MEAN) / MNIST_STD
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    per_class = cm.diagonal() / cm.sum(axis=1) * 100

    x_pos = np.arange(10) + i * 0.25
    ax.bar(x_pos, per_class, width=0.22, label=name,
           color=COLORS[name][0], alpha=0.85, edgecolor='grey', linewidth=0.5)

ax.set_xlabel('Digit', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Test Accuracy by Model', fontsize=14, fontweight='bold')
ax.set_xticks(np.arange(10) + 0.25)
ax.set_xticklabels(range(10))
ax.legend(fontsize=11)
ax.set_ylim(90, 101)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: per_class_accuracy.png')

## 8. Overfitting Analysis

Plot the gap between training and validation accuracy over epochs to visualize overfitting dynamics.

In [ ]:
# Overfitting Gap Over Epochs
fig, ax = plt.subplots(figsize=(10, 6))

for name, hist in histories.items():
    gap = [t - v for t, v in zip(hist['train_acc'], hist['val_acc'])]
    epochs = range(1, len(gap) + 1)
    ax.plot(epochs, gap, '-o', color=COLORS[name][0], markersize=4,
            label=name, linewidth=2)

ax.axhline(y=0, color='#57606a', linestyle='--', linewidth=0.8)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy Gap (Train - Val) %', fontsize=12)
ax.set_title('Overfitting Analysis: Train-Val Accuracy Gap Over Epochs',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('overfitting_gap.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: overfitting_gap.png')

## 9. Summary Dashboard

A single composite figure summarizing key results.

In [ ]:
# Summary Dashboard
fig = plt.figure(figsize=(20, 12))
fig.suptitle('Neural Network Regularization: Summary Dashboard',
             fontsize=18, fontweight='bold', y=0.98)

# 1. Loss curves (all on one)
ax1 = fig.add_subplot(2, 2, 1)
for name, hist in histories.items():
    c_train, c_val = COLORS[name]
    epochs = range(1, len(hist['train_loss']) + 1)
    ax1.plot(epochs, hist['train_loss'], '-',  color=c_train, linewidth=2, label=f'{name} (Train)')
    ax1.plot(epochs, hist['val_loss'],   '--', color=c_val,   linewidth=2, label=f'{name} (Val)')
ax1.set_title('Loss Curves', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend(fontsize=8, ncol=2)
ax1.grid(True, alpha=0.3)

# 2. Accuracy curves (all on one)
ax2 = fig.add_subplot(2, 2, 2)
for name, hist in histories.items():
    c_train, c_val = COLORS[name]
    epochs = range(1, len(hist['train_acc']) + 1)
    ax2.plot(epochs, hist['train_acc'], '-',  color=c_train, linewidth=2, label=f'{name} (Train)')
    ax2.plot(epochs, hist['val_acc'],   '--', color=c_val,   linewidth=2, label=f'{name} (Val)')
ax2.set_title('Accuracy Curves', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend(fontsize=8, ncol=2)
ax2.grid(True, alpha=0.3)

# 3. Train vs Test bar chart
ax3 = fig.add_subplot(2, 2, 3)
x = np.arange(len(model_names))
width = 0.30
ax3.bar(x - width/2, train_accs, width, label='Train',
        color=[COLORS[n][0] for n in model_names], alpha=0.85)
ax3.bar(x + width/2, test_accs,  width, label='Test',
        color=[COLORS[n][1] for n in model_names], alpha=0.85)
for i, (tr, te) in enumerate(zip(train_accs, test_accs)):
    ax3.text(i - width/2, tr + 0.2, f'{tr:.1f}', ha='center', fontsize=9, fontweight='bold')
    ax3.text(i + width/2, te + 0.2, f'{te:.1f}', ha='center', fontsize=9, fontweight='bold')
ax3.set_title('Train vs Test Accuracy', fontsize=13, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(model_names, fontsize=10)
ax3.set_ylabel('Accuracy (%)')
ax3.set_ylim(90, 101)
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# 4. Generalization gap
ax4 = fig.add_subplot(2, 2, 4)
gaps = results_df['Gap (Train-Test)'].tolist()
bars = ax4.bar(model_names, gaps, color=[COLORS[n][0] for n in model_names], alpha=0.85, width=0.5)
for bar, gap in zip(bars, gaps):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{gap:.2f}%', ha='center', fontsize=10, fontweight='bold')
ax4.set_title('Generalization Gap (Train - Test)', fontsize=13, fontweight='bold')
ax4.set_ylabel('Gap (%)')
ax4.axhline(y=0, color='#57606a', linestyle='--', linewidth=0.8)
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('summary_dashboard.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: summary_dashboard.png')

## 10. Discussion: How Regularization Improves Generalization

### What Is Overfitting?
Overfitting occurs when a neural network memorizes the training data (including noise) rather than learning generalizable patterns. A model that overfits achieves **high training accuracy** but **poor test accuracy**, where the gap between the two is the **generalization gap**.

### Baseline (No Regularization)
- The baseline model has no constraints on its weights. It is free to "memorize" training examples.
- As training progresses, the training loss drops continuously, but the validation loss eventually **starts increasing** — a classic sign of overfitting.
- The generalization gap (Train Acc - Test Acc) is the **largest** among the three models.

### Dropout Regularization (p = 0.5)
- **How it works:** During each forward pass in training, Dropout randomly **zeroes out 50% of neurons** in each hidden layer. This forces the network to learn **redundant, distributed representations** rather than relying on any single neuron or co-adaptation of neurons.
- **Effect on training:** Training accuracy is **lower** than the baseline because the model is effectively training a different sub-network each iteration.
- **Effect on testing:** At test time, all neurons are active (with scaled weights), effectively **ensembling** many sub-networks. This produces **better generalization** and a **smaller gap** between training and test accuracy.
- **Intuition:** Dropout acts as an implicit ensemble of $2^n$ thinned networks (where $n$ is the number of dropout-eligible neurons).

### L2 Regularization (Weight Decay, λ = 1e-3)
- **How it works:** L2 regularization adds a penalty term $\frac{\lambda}{2} \|\mathbf{w}\|_2^2$ to the loss function. This **discourages large weights** by shrinking them toward zero.
- **Effect:** The network prefers **simpler, smoother** decision boundaries because large weights (which create sharp, complex boundaries) are penalized.
- **Result:** Training accuracy is slightly lower than the unregularized baseline, but the test accuracy improves because the model's decision surface generalizes better to unseen data.
- **Mathematically:** The gradient update becomes:
  $$w \leftarrow w - \eta \left( \nabla_{w} \mathcal{L} + \lambda w \right) = (1 - \eta \lambda) w - \eta \nabla_{w} \mathcal{L}$$
  This $(1 - \eta \lambda)$ factor is why L2 regularization is called **weight decay**.

### Summary

| Aspect | Baseline | Dropout | L2 Reg |
|--------|----------|---------|--------|
| Training Accuracy | **Highest** | Lower | Slightly Lower |
| Test Accuracy | Lowest | **Higher** | **Higher** |
| Generalization Gap | **Largest** | **Smallest** | Small |
| Mechanism | None | Random neuron masking | Weight penalty |
| Effect | Memorizes data | Learns robust features | Prefers simple models |

**Key Takeaway:** Regularization techniques reduce overfitting by constraining the model's capacity. Dropout does this by forcing distributed representations, while L2 does this by penalizing the magnitude of weights. Both lead to **better generalization** (i.e., improved test accuracy relative to training accuracy).

In [ ]:
# Classification Reports
for name in model_configs:
    model = models[name]
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = (images - MNIST_MEAN) / MNIST_STD
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f'\n{"="*60}')
    print(f'Classification Report: {name}')
    print(f'{"="*60}')
    print(classification_report(all_labels, all_preds, digits=4))

In [ ]:
print('\n✅ Notebook complete. All plots saved to disk.')
print('   Saved files: loss_curves.png, accuracy_curves.png,')
print('   accuracy_comparison.png, generalization_gap.png,')
print('   confusion_matrices.png, per_class_accuracy.png,')
print('   overfitting_gap.png, summary_dashboard.png')